<a href="https://colab.research.google.com/github/xidoudou/ai-agents/blob/main/agent02_finance_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Install & Import**

In [ ]:
!pip install openai langgraph langchain_openai langchain_core
import re, os, requests
from openai import OpenAI
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
EXCHANGE_RATE_API_KEY = userdata.get("EXCHANGERATE_API_KEY")
client = OpenAI()

**Functions & Tools**

In [ ]:
from langchain_core.tools import tool

@tool
def calculate(what):
  """Evaluates a math expression, e.g. '4 * 7 + 2 """
  return eval(what)


@tool
def get_exchange_rate(from_currency, to_currency):
  """
  Get the exchange rate from one currency to another.
  Currency codes should be 3-letter codes like USD, EUR, GBP.
  """
  API_key = EXCHANGE_RATE_API_KEY
  url = f"https://v6.exchangerate-api.com/v6/{API_key}/latest/{from_currency}"
  response = requests.get(url)
  data = response.json()
  if data["result"] != "success":
    return f"Error: could not fetch exchange rate for {from_currency}"
  if to_currency not in data["conversion_rates"]:
    return f"Error: unknown currency code {to_currency}"
  return data["conversion_rates"][to_currency]



**Agent (Langgraph)**


In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI

class AgentState(TypedDict):
  messages: Annotated[list[AnyMessage], operator.add]


class Agent:
  def __init__(self, model, tools, system = ""):
    self.system = system
    graph = StateGraph(AgentState)
    graph.add_node("llm", self.call_openai)
    graph.add_node("action", self.take_action)
    graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
    graph.add_edge("action", "llm")
    graph.set_entry_point("llm")
    self.graph = graph.compile()
    self.tools = {t.name: t for t in tools}
    self.model = model.bind_tools(tools)

  def exists_action(self, state):
    result = state['messages'][-1]
    return len(result.tool_calls) > 0

  def call_openai(self, state):
    messages = state['messages']
    if self.system:
      messages = [SystemMessage(content=self.system)] + messages
    message = self.model.invoke(messages)
    return {'messages': [message]}

  def take_action(self, state):
    tool_calls = state['messages'][-1].tool_calls
    results = []
    for t in tool_calls:
      print(f"Calling: {t}")
      result = self.tools[t['name']].invoke(t['args'])
      results.append(ToolMessage(tool_call_id=t['id'], name = t['name'], content = str(result)))
    print("Back to the model!")
    return {'messages': results}



**Prompt**

In [ ]:
prompt = """
You are a helpful financial assistant.
Use the tools available to you to answer questions about currency exchange and calculations.
Only use a tool when you need to look up information or do a calculation other wise answer directly.

When giving your final answer, use plain text only (no markdown, no bold,
no bullet points). Keep it to one or two clear sentences.
""".strip()

**Test**

In [ ]:
model = ChatOpenAI(model="gpt-6-astra")
finance_bot = Agent(model, [calculate, get_exchange_rate], system = prompt)

In [ ]:
from langchain_core.messages import HumanMessage

messages = [HumanMessage(content="I have 50 euro and 30 rmb and 50 dollar, how much in total in us dollar?")]
result = finance_bot.graph.invoke({"messages": messages})
print(result['messages'][-1].content)